#### Attention layer based Machine learning based multi-class Skin Cancer Diagnosis

### STEP 1 - Import library & Setup

In [1]:
!pip install torch torchvision

import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm
from tqdm import tqdm
from PIL import Image

### STEP 2 - Data Import from Drive

In [2]:
import os
import pandas as pd

BASE_DIR = "/Users/abdurrahman/Downloads/Skin_Cancer"

metadata_path = os.path.join(BASE_DIR, "HAM10000_metadata.csv")
metadata = pd.read_csv(metadata_path)

print("Metadata shape:", metadata.shape)

Metadata shape: (10015, 7)


### STEP 2 - Data Import from Drive

In [3]:
import os

print("Files inside BASE_DIR:")
print(os.listdir(BASE_DIR))

Files inside BASE_DIR:
['.DS_Store', 'HAM10000_images_part_1', 'HAM10000_images_part_2', 'HAM10000_metadata.csv']


### STEP 3 - Device Configuration

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


### STEP 4 - Dataset Paths

In [5]:
for part in ['HAM10000_images_part_1', 'HAM10000_images_part_2']:
    src = f'/content/dataset/{part}'
    if os.path.exists(src):
        for fname in os.listdir(src):
            shutil.copy(os.path.join(src, fname), base_path)

### STEP 5 - Prepare dataframe

In [6]:
metadata_path = os.path.join(BASE_DIR, "HAM10000_metadata.csv")
metadata = pd.read_csv(metadata_path)

print("Metadata loaded:", metadata.shape)

class_map = {
    'akiec': 0,
    'bcc': 1,
    'bkl': 2,
    'df': 3,
    'mel': 4,
    'nv': 5,
    'vasc': 6
}

metadata['label'] = metadata['dx'].map(class_map)

def find_image_path(image_id):
    part1 = os.path.join(BASE_DIR, "HAM10000_images_part_1", f"{image_id}.jpg")
    part2 = os.path.join(BASE_DIR, "HAM10000_images_part_2", f"{image_id}.jpg")

    if os.path.exists(part1):
        return part1
    elif os.path.exists(part2):
        return part2
    else:
        return None

metadata["base_path"] = metadata["image_id"].apply(find_image_path)

# Drop missing files
metadata = metadata.dropna(subset=["base_path"])

print("Final dataset size:", len(metadata))
print(metadata["label"].value_counts(normalize=True))

Metadata loaded: (10015, 7)
Final dataset size: 10015
label
5    0.669496
4    0.111133
2    0.109735
1    0.051323
0    0.032651
6    0.014179
3    0.011483
Name: proportion, dtype: float64


### STEP 6 - Train/Val/Test Split + Oversampling (training only)

In [7]:
train_df, temp_df = train_test_split(metadata, test_size=0.3, stratify=metadata['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(train_df[['base_path']], train_df['label'])
train_balanced = pd.concat([X_res, pd.Series(y_res, name='label')], axis=1)

print(f"Balanced train size: {len(train_balanced)}")

Balanced train size: 32851


### STEP 7 - Custom Dataset Class

In [8]:
class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'base_path']
        label = self.df.loc[idx, 'label']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

print('Done')

Done


### STEP 8 - Data Transforms & Loaders

In [9]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = SkinCancerDataset(train_balanced, train_transform)
val_ds   = SkinCancerDataset(val_df, val_test_transform)
test_ds  = SkinCancerDataset(test_df, val_test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

### STEP 9 - Data Transforms & Loaders

In [10]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_planes, in_planes // ratio, bias=False),
            nn.ReLU(),
            nn.Linear(in_planes // ratio, in_planes, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x).squeeze(-1).squeeze(-1))
        max_out = self.fc(self.max_pool(x).squeeze(-1).squeeze(-1))
        out = avg_out + max_out
        return self.sigmoid(out).unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv(x)
        return self.sigmoid(x)

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

### STEP 10 - Model Builder

In [11]:
NUM_CLASSES = 7

def build_model(base_name, use_attention=False):
    if base_name == "simple_cnn":
        model = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, NUM_CLASSES)
        )
    else:
        model = timm.create_model(base_name, pretrained=True, num_classes=NUM_CLASSES)
        # Freeze early layers (optional fine-tuning strategy)
        for param in list(model.parameters())[:-20]:
            param.requires_grad = False

    if use_attention:
        # Insert CBAM after the feature extractor (before classifier head)
        # For timm models → replace the forward_features + head logic
        class AttnWrapper(nn.Module):
            def __init__(self, base, attn):
                super().__init__()
                self.base = base
                self.attn = attn
                self.head = base.get_classifier()

            def forward(self, x):
                x = self.base.forward_features(x)
                x = self.attn(x)
                x = self.base.global_pool(x)
                x = self.head(x)
                return x

        model = AttnWrapper(model, CBAM(model.num_features))

    return model.to(device)

### STEP 11 - Model Builder

In [ ]:
import os

models_to_train = {
    "SimpleCNN": build_model("simple_cnn", False),
    "EfficientNetB0": build_model("efficientnet_b0", False),
    "EfficientNetB0_Attn": build_model("efficientnet_b0", True),
    "EfficientNetB3": build_model("efficientnet_b3", False),
    "EfficientNetB3_Attn": build_model("efficientnet_b3", True),
}

criterion = nn.CrossEntropyLoss()
results = {}

# Create the models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

for name, model in models_to_train.items():
    print(f"\n=== Training {name} ===")
    optimizer = optim.Adam(model.parameters(), lr=1e-3 if "Attn" in name else 3e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

    best_val_acc = 0
    for epoch in range(20):  # increase to 30-50 later
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for imgs, labels in tqdm(train_loader):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_acc = train_correct / train_total
        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} Accuracy: {train_acc:.4f} | Val Loss: {val_loss:.4f} Accuracy: {val_acc:.4f}")

        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"models/{name}_best.pth")

    results[name] = best_val_acc


=== Training SimpleCNN ===


100%|████████████████████████████████████████████████████████████████████████████████████| 1027/1027 [09:34<00:00,  1.79it/s]


Epoch  1 | Train Loss: 1.6485 Accuracy: 0.3502 | Val Loss: 1.2282 Accuracy: 0.5107


100%|████████████████████████████████████████████████████████████████████████████████████| 1027/1027 [09:24<00:00,  1.82it/s]


Epoch  2 | Train Loss: 1.3746 Accuracy: 0.4684 | Val Loss: 1.3315 Accuracy: 0.4727


...

100%|████████████████████████████████████████████████████████████████████████████████████| 1027/1027 [30:13<00:00,  1.77s/it]


Epoch  6 | Train Loss: 0.1026 Accuracy: 0.9634 | Val Loss: 0.6515 Accuracy: 0.8089


100%|████████████████████████████████████████████████████████████████████████████████████| 1027/1027 [30:15<00:00,  1.77s/it]


Epoch  7 | Train Loss: 0.0885 Accuracy: 0.9689 | Val Loss: 0.6312 Accuracy: 0.8123


  3%|██▍                                                                                   | 29/1027 [00:52<32:32,  1.96s/it]

### STEP 12 - Final Test Evaluation

In [ ]:
print("\n=== Test Results ===")
test_results = []

for name, model in models_to_train.items():
    model.load_state_dict(torch.load(f"models/{name}_best.pth"))
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            y_pred.extend(preds.cpu().numpy())
            y_true.extend(labels.numpy())

    acc = np.mean(np.array(y_true) == np.array(y_pred))
    print(f"{name} Test Accuracy: {acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=list(class_map.keys())))
    test_results.append([name, acc])

df_results = pd.DataFrame(test_results, columns=['Model', 'Test Acc'])
print(df_results.sort_values('Test Accuracy', ascending=False))